# B2 — `C.Oc138k.compact3.e4 → Oc`: Model 2 without the yield field, trained to convergence

Two changes against B1, both decided by measurement rather than by taste.

**1. `yield_percent` is dropped from the target.** It is not sparse: 60.7% of the 138,869
training rows carry a yield, against 15.9% for catalyst — the field the model predicts best.
What B1 did with it is the argument: of 5,687 test records its rank-1 generation emitted a
yield number 61 times and wrote `?` 5,234 times. The model was not mispredicting the yield,
it had learned that abstaining beats any guess, which is the correct thing to learn when the
target is not a function of the input. A yield depends on scale, purity, work-up and
operator; none of that is in `(product, reactants)`. Keeping the field costs target length
and teaches the decoder to emit a placeholder.

Temperature stays, on the same evidence read the other way: it is populated in only 27.5% of
rows, yet B1 produced real numbers for it (relaxed top-5 21.8%). Weak, but a signal.

**2. Four epochs instead of three, and a budget that reaches them.** Both previous attempts
were cut by the clock with `eval_loss` still falling monotonically — B1 v2 at epoch 1.728 of
3, B1 v3 at 2.189 of 3 (4.641 → 1.047, no upturn anywhere). The comparison against t5-small
was therefore never on equal footing. At the measured ~103 min/epoch for the four-field
target, 400 minutes covers roughly 4 epochs of the shorter three-field one, and
`load_best_model_at_end` takes the minimum wherever it falls.

**What this run is for.** B1 v3 split by field: catalyst clearly better than t5-small
(23.8/34.1/53.9 against 22.3/31.5/39.8 for strict top-1 / strict top-5 / relaxed top-5),
temperature slightly better (4.4/15.5/21.8 against 3.6/14.9/17.0), solvent clearly worse
(18.9/33.1/50.8 against 29.5/49.5/61.3). Reading: the chemical base pays on long
organometallic SMILES and loses on the few dozen short frequent solvent strings, where the
task is closer to classification over a small label set. That reading only holds if the
undertraining is removed, which is what this run does.

The matching t5-small run (`B2-t5small`) uses the same data, the same three fields and the
same compact serialization, so the base is the only difference between them. The t5-small
numbers quoted above come from JSON targets and are *not* a clean control for that reason.

**Per-record output is kept**, not just the summary: `--output` writes every record's beam
list, which is what a paired McNemar between the two bases needs. B1 could not be tested that
way because the t5-small runs survive only as prose numbers in RESULTS.md.

**Data:** `kuzmenkoiryna/retro-planner-ord-conditions` — 138,869 train + 7,714 val, and the
leak-free 5,687-record test set.

**Cost:** ~6 h 40 min training + ~40 min evaluation.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/v2_ord_conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4   # same as every earlier Model 2 run
condition_fields = "solvent,catalyst,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_compoundt5_3f"
# ~103 min/epoch measured on the four-field target; the three-field one is shorter, so
# 400 minutes should cover all four epochs. The cap is a guard against losing the session,
# not the plan: `load_best_model_at_end` keeps the best checkpoint wherever training stops.
time_budget_minutes = 400

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Batch 8 x 4 accumulation, not the script's default 32 x 1: that default was set for
# t5-small (60M) and CompoundT5 is 220M, which put both T4s at 14.55 of 14.56 GiB and
# raised `torch.OutOfMemoryError` a few steps in. The effective batch stays 32, so the
# optimization matches the t5-small runs -- only the memory footprint moves.
#
# --max-target-length 200 for the same reason as B1: catalysts are whole organometallic
# complexes and reach 187 tokens at p99 under this vocabulary. Dropping the yield field
# shortens the tail by a few tokens only, so the cap does not move.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_work \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --max-target-length 200 \
    --per-device-train-batch-size 8 \
    --per-device-eval-batch-size 8 \
    --gradient-accumulation-steps 4 \
    --learning-rate {learning_rate} \
    --num-train-epochs 4 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The vocabulary repair must have fired: CompoundT5's 221-token ZINC vocabulary cannot spell
# `|`, `?` or several digits. Without it the targets would be <unk>-corrupted, which is what
# sank the earlier ReactionT5-based attempt at a chemical base under Model 2.
!grep -E "new character token|Train examples|condition field" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
marker = json.load(open(f"{output_dir}/final/conditions_format.json"))
print("format marker:", marker)
assert marker["fields"] == condition_fields.split(","), "marker disagrees with the requested fields"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))
# The question B1 could not answer: did the curve turn, or did the clock stop it again?
best_epoch = min(points, key=lambda p: p[1])[0]
print(f"  best at epoch {best_epoch:.2f} of {points[-1][0]:.2f} reached")

In [ ]:
# --target-format and the field list both default to the marker written next to the
# checkpoint. Batch 8 for the same reason as training: beam 10 over a 220M model holds 10
# sequences per record in memory, where the t5-small runs used 32.
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 8 --device cuda \
    --max-target-length 200 \
    --output "/kaggle/working/B2_conditions_compoundt5_3f_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/B2_conditions_compoundt5_3f_clean_topk.json"))
print(json.dumps(data["summary"], indent=2))
print("per-record entries kept for a paired test:", len(data["records"]))